## montly analysis of the measles dataset

In [1]:
# import the  libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np      
import os
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# load the data
monthly_data = pd.read_csv("../data/cases_month.csv")
monthly_data.head()


,region,country,iso3,year,month,measles_suspect,measles_clinical,measles_epi_linked,measles_lab_confirmed,measles_total,rubella_clinical,rubella_epi_linked,rubella_lab_confirmed,rubella_total,discarded
0,AFR,Algeria,DZA,2012,1,8.0,6.0,0.0,2.0,8.0,NaN,NaN,NaN,NaN,0.0
1,AFR,Algeria,DZA,2012,2,10.0,10.0,0.0,0.0,10.0,NaN,NaN,NaN,NaN,0.0
2,AFR,Algeria,DZA,2012,3,17.0,17.0,0.0,0.0,17.0,NaN,NaN,NaN,NaN,0.0
3,AFR,Algeria,DZA,2012,4,7.0,5.0,0.0,0.0,5.0,0.0,0.0,1.0,1.0,2.0
4,AFR,Algeria,DZA,2012,5,14.0,11.0,0.0,0.0,11.0,0.0,0.0,3.0,3.0,3.0


In [4]:
monthly_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 22780 entries, 0 to 22779
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   region                 22780 non-null  str    
 1   country                22780 non-null  str    
 2   iso3                   22780 non-null  str    
 3   year                   22780 non-null  int64  
 4   month                  22780 non-null  int64  
 5   measles_suspect        22632 non-null  float64
 6   measles_clinical       22632 non-null  float64
 7   measles_epi_linked     22632 non-null  float64
 8   measles_lab_confirmed  22632 non-null  float64
 9   measles_total          22632 non-null  float64
 10  rubella_clinical       7811 non-null   float64
 11  rubella_epi_linked     7811 non-null   float64
 12  rubella_lab_confirmed  7811 non-null   float64
 13  rubella_total          7811 non-null   float64
 14  discarded              22632 non-null  float64
dtypes: float64(10

| Variable | Class | Description |
|---|---|---|
| `region` | character | Region name |
| `country` | character | Country name |
| `iso3` | character | Three-letter country code |
| `year` | double | Year |
| `month` | double | Month |
| `measles_suspect` | double | Suspected measles cases: a patient with fever and maculopapular (non-vesicular) rash, or a case suspected by a health-care worker |
| `measles_clinical` | double | Clinically compatible measles cases: suspected case with fever and maculopapular rash and at least one of cough, coryza, or conjunctivitis; no adequate specimen and not epidemiologically linked to a lab-confirmed case |
| `measles_epi_linked` | double | Epidemiologically linked measles cases: suspected measles case not lab-confirmed but geographically and temporally related (rash onset 7–23 days apart) to a lab-confirmed or epidemiologically linked case |
| `measles_lab_confirmed` | double | Laboratory-confirmed measles cases: suspected measles case confirmed positive in a proficient laboratory, with vaccine-associated illness ruled out |
| `measles_total` | double | Total measles cases: sum of clinically compatible, epidemiologically linked, and laboratory-confirmed cases |
| `rubella_clinical` | double | Clinically compatible rubella cases |
| `rubella_epi_linked` | double | Epidemiologically linked rubella cases |
| `rubella_lab_confirmed` | double | Laboratory-confirmed rubella cases |
| `rubella_total` | double | Total rubella cases |
| `discarded` | double | Discarded cases: suspected cases investigated and discarded as non-measles (and non-rubella) |

## data cleaning

In [5]:
#standardize column names
monthly_data.columns = monthly_data.columns.str.lower().str.replace(" ", "_")
monthly_data.head()

,region,country,iso3,year,month,measles_suspect,measles_clinical,measles_epi_linked,measles_lab_confirmed,measles_total,rubella_clinical,rubella_epi_linked,rubella_lab_confirmed,rubella_total,discarded
0,AFR,Algeria,DZA,2012,1,8.0,6.0,0.0,2.0,8.0,NaN,NaN,NaN,NaN,0.0
1,AFR,Algeria,DZA,2012,2,10.0,10.0,0.0,0.0,10.0,NaN,NaN,NaN,NaN,0.0
2,AFR,Algeria,DZA,2012,3,17.0,17.0,0.0,0.0,17.0,NaN,NaN,NaN,NaN,0.0
3,AFR,Algeria,DZA,2012,4,7.0,5.0,0.0,0.0,5.0,0.0,0.0,1.0,1.0,2.0
4,AFR,Algeria,DZA,2012,5,14.0,11.0,0.0,0.0,11.0,0.0,0.0,3.0,3.0,3.0


In [6]:
#check for duplictes
monthly_data.duplicated().sum()

np.int64(0)

In [ ]:
# Data cleaning pipeline (monthly measles)
region_map = {
    "AFR": "AFRO",
    "AMR": "AMRO",
    "EMR": "EMRO",
    "EUR": "EURO",
    "SEAR": "SEARO",
    "WPR": "WPRO",
}

monthly_clean = monthly_data.copy()

# Standardize text fields
for col in ["region", "country", "iso3"]:
    monthly_clean[col] = monthly_clean[col].astype("string").str.strip()

# Normalize region codes to WHO naming used in cleaned outputs
monthly_clean["region"] = monthly_clean["region"].replace(region_map)

# Coerce numeric columns to numeric dtype
numeric_cols = [
    "year",
    "month",
    "measles_suspect",
    "measles_clinical",
    "measles_epi_linked",
    "measles_lab_confirmed",
    "measles_total",
    "rubella_clinical",
    "rubella_epi_linked",
    "rubella_lab_confirmed",
    "rubella_total",
    "discarded",
]
monthly_clean[numeric_cols] = monthly_clean[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
 )

# Keep valid months only (1-12)
monthly_clean = monthly_clean[monthly_clean["month"].between(1, 12)]

# Build helper date column from year/month
monthly_clean["report_date"] = pd.to_datetime(
    monthly_clean["year"].astype("Int64").astype("string")
    + "-"
    + monthly_clean["month"].astype("Int64").astype("string").str.zfill(2)
    + "-01",
    errors="coerce",
)

monthly_clean.head()

In [ ]:
# Replace invalid negative surveillance counts with missing values
count_cols = [
    "measles_suspect",
    "measles_clinical",
    "measles_epi_linked",
    "measles_lab_confirmed",
    "measles_total",
    "rubella_clinical",
    "rubella_epi_linked",
    "rubella_lab_confirmed",
    "rubella_total",
    "discarded",
]

monthly_clean[count_cols] = monthly_clean[count_cols].mask(
    monthly_clean[count_cols] < 0
)

negatives_remaining = (monthly_clean[count_cols] < 0).sum().sum()
print(f"Negative values remaining in count columns: {int(negatives_remaining)}")

In [ ]:
# Remove duplicate records and inspect missingness
before_rows = len(monthly_clean)
monthly_clean = monthly_clean.drop_duplicates()
after_rows = len(monthly_clean)

print(f"Rows before deduplication: {before_rows}")
print(f"Rows after deduplication:  {after_rows}")
print(f"Duplicates removed:        {before_rows - after_rows}")

monthly_clean.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
# Save cleaned monthly data
output_path = "../data/cleaned/cases_month_clean.csv"
monthly_clean.to_csv(output_path, index=False)
print(f"Saved cleaned data to: {output_path}")